In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
import warnings
from tabulate import tabulate
import pickle
import os

warnings.filterwarnings('ignore')

# --- Custom Metrics (Same as XGB) ---
def mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def nse(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))

def rpd(y_true, y_pred):
    return np.std(y_true) / np.std(y_true - y_pred)

def aic(y_true, y_pred, k):
    resid = y_true - y_pred
    sse = np.sum(resid ** 2)
    n = len(y_true)
    return n * np.log(sse / n) + 2 * k

# --- Load Data ---
try:
    df = pd.read_excel("CCCdataset4.xlsx")
except FileNotFoundError:
    print("Error: CCCdataset.xlsx not found. Please ensure the file is in the correct directory.")
    raise

input_vars = ['CC', 'FA', 'CA', 'WC', 'P']
output_vars = ['pc', 'fc', 'Ec', 'e']

# --- Plot Settings ---
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 9

# --- Storage for models and scalers (Added for MOO compatibility) ---
bpnn_models = {}
bpnn_x_scalers = {}
bpnn_y_scalers = {}
performance_df = pd.DataFrame(columns=['Target', 'RMSE', 'MAPE', 'NSE', 'R2_test', 'R2_train', 'RPD', 'AIC', 'TrainSize', 'TestSize'])
all_best_params = []

# --- Optuna Objective with K-fold (Consistent with XGB) ---
def objective(trial, X, y):
    n_layers = trial.suggest_int('n_layers', 1, 3)
    hidden_layer_sizes = tuple([trial.suggest_int(f'n_units_l{i}', 10, 100) for i in range(n_layers)])
    params = {
        'hidden_layer_sizes': hidden_layer_sizes,
        'activation': trial.suggest_categorical('activation', ['relu', 'tanh']),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 1e-1, log=True),
        'alpha': trial.suggest_float('alpha', 1e-5, 1e-1, log=True),
        'max_iter': 1000,
        'random_state': 42
    }

    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse_scores = []
    
    for train_idx, val_idx in kfold.split(X):
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        model = MLPRegressor(**params)
        model.fit(X_train_fold, y_train_fold)

        val_pred = model.predict(X_val_fold)
        rmse = np.sqrt(mean_squared_error(y_val_fold, val_pred))
        rmse_scores.append(rmse)
    
    return np.mean(rmse_scores)

# --- Main Loop (Same structure as XGB with model storage added) ---
for target in output_vars:
    df_clean = df[input_vars + [target]].dropna(subset=[target])

    if len(df_clean) < 10:
        print(f"Skipping target {target} due to insufficient data.")
        continue

    X = df_clean[input_vars]
    y = df_clean[target].values.reshape(-1, 1)

    # --- Scale Data ---
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y).ravel()

    # --- Split Data (80/20) ---
    X_train_full, X_test, y_train_full, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    # --- Hyperparameter Optimization ---
    study = optuna.create_study(direction='minimize')
    study.optimize(lambda trial: objective(trial, X_train_full, y_train_full), n_trials=50, show_progress_bar=True)

    # --- Get Best Params ---
    best_params = study.best_params
    # Remove n_layers from best_params since it's not an MLP parameter
    n_layers = best_params.pop('n_layers')
    # Create hidden_layer_sizes tuple
    best_params['hidden_layer_sizes'] = tuple([best_params.pop(f'n_units_l{i}') for i in range(n_layers)])
    best_params.update({
        'max_iter': 1000,
        'random_state': 42
    })
    all_best_params.append({'Target': target, 'n_layers': n_layers, **best_params})

    # --- Train Final Model on Full Training Data ---
    final_model = MLPRegressor(**best_params)
    final_model.fit(X_train_full, y_train_full)
    
    # --- Store models and scalers for MOO compatibility ---
    bpnn_models[target] = final_model
    bpnn_x_scalers[target] = scaler_X
    bpnn_y_scalers[target] = scaler_y

    # --- Predictions ---
    y_pred_test_scaled = final_model.predict(X_test)
    y_pred_train_scaled = final_model.predict(X_train_full)

    # --- Inverse Scaling ---
    y_pred_test = scaler_y.inverse_transform(y_pred_test_scaled.reshape(-1, 1)).ravel()
    y_pred_train = scaler_y.inverse_transform(y_pred_train_scaled.reshape(-1, 1)).ravel()
    y_test_original = scaler_y.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_train_full_original = scaler_y.inverse_transform(y_train_full.reshape(-1, 1)).ravel()

    # --- Calculate Metrics (Same as XGB) ---
    performance_df = pd.concat([performance_df, pd.DataFrame([{
        'Target': target,
        'RMSE': np.sqrt(mean_squared_error(y_test_original, y_pred_test)),
        'MAPE': mape(y_test_original, y_pred_test),
        'NSE': nse(y_test_original, y_pred_test),
        'R2_test': r2_score(y_test_original, y_pred_test),
        'R2_train': r2_score(y_train_full_original, y_pred_train),
        'RPD': rpd(y_test_original, y_pred_test),
        'AIC': aic(y_test_original, y_pred_test, k=X.shape[1]),
        'TrainSize': len(X_train_full),
        'TestSize': len(X_test)
    }])], ignore_index=True)

# --- Results Display (Same format as XGB) ---
print("\n=== Performance Metrics ===")
print(performance_df)

best_params_df = pd.DataFrame(all_best_params)
pd.set_option('display.max_columns', None)
print("\n=== Best Hyperparameters for Each Target ===")
print(best_params_df)

# --- Save models and scalers to pickle file (UPDATED TO MATCH YOUR LOADING PATTERN) ---
def save_bpnn_models(models_dict, x_scalers_dict, y_scalers_dict, filename="concrete_models.pkl"):
    """
    Save BPNN models and scalers to a pickle file
    Updated to match the expected loading pattern
    """
    # Create a dictionary containing all objects to save with exact key names
    save_data = {
        'bpnn_models': models_dict,
        'bpnn_x_scalers': x_scalers_dict,
        'bpnn_y_scalers': y_scalers_dict,
        'input_vars': input_vars,
        'output_vars': output_vars,
        'best_params': all_best_params,
        'performance_metrics': performance_df.to_dict()
    }
    
    # Save to pickle file
    with open(filename, 'wb') as f:
        pickle.dump(save_data, f)
    
    print(f"\n=== Models and scalers saved to {filename} ===")
    print(f"File size: {os.path.getsize(filename) / 1024:.2f} KB")
    print(f"Models saved for targets: {list(models_dict.keys())}")
    print(f"Keys in saved file: {list(save_data.keys())}")

# --- Load models and scalers from pickle file (UPDATED TO MATCH YOUR PATTERN) ---
def load_bpnn_models(filename="concrete_models.pkl"):
    """
    Load BPNN models and scalers from a pickle file
    Updated to match your exact loading pattern
    """
    try:
        with open(filename, 'rb') as f:
            model_data = pickle.load(f)
        
        # Extract using the exact key names from your loading code
        bpnn_models = model_data['bpnn_models']
        bpnn_x_scalers = model_data['bpnn_x_scalers']
        bpnn_y_scalers = model_data['bpnn_y_scalers']
        input_vars = model_data['input_vars']
        output_vars = model_data['output_vars']
        
        print(f"\n=== Models and scalers loaded from {filename} ===")
        print(f"Available targets: {list(bpnn_models.keys())}")
        
        return bpnn_models, bpnn_x_scalers, bpnn_y_scalers, input_vars, output_vars
                
    except FileNotFoundError:
        raise Exception("Model file not found. Please run the training notebook first and export models.")
    except KeyError as e:
        print(f"KeyError: {e}. Available keys in file: {list(model_data.keys())}")
        raise

# --- Predict using loaded models ---
def predict_with_loaded_models(loaded_data, new_data):
    """
    Make predictions using loaded models
    """
    # Unpack the loaded data
    bpnn_models, bpnn_x_scalers, bpnn_y_scalers, input_vars, output_vars = loaded_data
    
    # Ensure new_data is a DataFrame with correct column names
    if not isinstance(new_data, pd.DataFrame):
        new_data = pd.DataFrame(new_data, columns=input_vars)
    
    predictions = {}
    
    for target in output_vars:
        if target in bpnn_models:
            # Scale input data
            X_scaled = bpnn_x_scalers[target].transform(new_data)
            
            # Make prediction
            y_pred_scaled = bpnn_models[target].predict(X_scaled)
            
            # Inverse scale prediction
            y_pred = bpnn_y_scalers[target].inverse_transform(
                y_pred_scaled.reshape(-1, 1)
            ).ravel()
            
            predictions[target] = y_pred
    
    return predictions

# --- Save your trained models ---
save_bpnn_models(bpnn_models, bpnn_x_scalers, bpnn_y_scalers, "concrete_models.pkl")

# --- Example of how to use the loaded models ---
print("\n=== Example: Loading and using saved models ===")

# Load the models using the updated function
loaded_data = load_bpnn_models("concrete_models.pkl")

# Unpack the loaded data
bpnn_models_loaded, bpnn_x_scalers_loaded, bpnn_y_scalers_loaded, input_vars_loaded, output_vars_loaded = loaded_data

# Create some sample data for prediction (using the first row of original data)
sample_data = df[input_vars].iloc[:1].copy()

print(f"Sample input data:")
print(sample_data)

# Make predictions
predictions = predict_with_loaded_models(loaded_data, sample_data)

print("\nPredictions:")
for target, pred in predictions.items():
    print(f"  {target}: {pred[0]:.4f}")

# --- Test the exact loading pattern from your code ---
print("\n=== Testing exact loading pattern from your code ===")
try:
    with open('concrete_models.pkl', 'rb') as f:
        model_data = pickle.load(f)
    
    bpnn_models_test = model_data['bpnn_models']
    bpnn_x_scalers_test = model_data['bpnn_x_scalers']
    bpnn_y_scalers_test = model_data['bpnn_y_scalers']
    input_vars_test = model_data['input_vars']
    output_vars_test = model_data['output_vars']
    
    print("Models loaded successfully using your exact pattern!")
    print(f"Available keys: {list(model_data.keys())}")
    
except Exception as e:
    print(f"Error: {e}")

print("\n=== Process completed successfully! ===")
print("You can now use 'concrete_models.pkl' in other scripts for predictions.")
print("The file uses the exact key names expected by your loading code.")

[I 2025-09-01 18:06:40,236] A new study created in memory with name: no-name-76f0b272-8dd5-4a14-a1dd-3d026a8f1649


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-01 18:06:40,515] Trial 0 finished with value: 0.24911599443690952 and parameters: {'n_layers': 2, 'n_units_l0': 48, 'n_units_l1': 37, 'activation': 'relu', 'learning_rate_init': 0.0005515328048249728, 'alpha': 0.05256726462502012}. Best is trial 0 with value: 0.24911599443690952.
[I 2025-09-01 18:06:40,597] Trial 1 finished with value: 0.22992245167228118 and parameters: {'n_layers': 1, 'n_units_l0': 15, 'activation': 'relu', 'learning_rate_init': 0.029205849736538177, 'alpha': 0.00012640061941818646}. Best is trial 1 with value: 0.22992245167228118.
[I 2025-09-01 18:06:41,139] Trial 2 finished with value: 0.24711359658103965 and parameters: {'n_layers': 3, 'n_units_l0': 94, 'n_units_l1': 27, 'n_units_l2': 43, 'activation': 'tanh', 'learning_rate_init': 0.0005395668691191647, 'alpha': 0.016102808713259954}. Best is trial 1 with value: 0.22992245167228118.
[I 2025-09-01 18:06:41,325] Trial 3 finished with value: 0.22064708925483334 and parameters: {'n_layers': 3, 'n_units_l0'

[I 2025-09-01 18:06:56,225] A new study created in memory with name: no-name-0540db78-f6b1-43eb-97df-ec1ebc03b921


[I 2025-09-01 18:06:56,169] Trial 49 finished with value: 0.21539557697747425 and parameters: {'n_layers': 3, 'n_units_l0': 56, 'n_units_l1': 66, 'n_units_l2': 91, 'activation': 'relu', 'learning_rate_init': 0.0050577549053212704, 'alpha': 3.073951468828864e-05}. Best is trial 49 with value: 0.21539557697747425.


...

[I 2025-09-01 18:07:46,227] Trial 0 finished with value: 0.45542049723642763 and parameters: {'n_layers': 3, 'n_units_l0': 92, 'n_units_l1': 19, 'n_units_l2': 24, 'activation': 'relu', 'learning_rate_init': 0.00019285634926491637, 'alpha': 0.0149404927912994}. Best is trial 0 with value: 0.45542049723642763.
[I 2025-09-01 18:07:47,434] Trial 1 finished with value: 0.4279733053460041 and parameters: {'n_layers': 2, 'n_units_l0': 97, 'n_units_l1': 60, 'activation': 'relu', 'learning_rate_init': 0.0003568689027434414, 'alpha': 0.005574004154103422}. Best is trial 1 with value: 0.4279733053460041.
[I 2025-09-01 18:07:48,315] Trial 2 finished with value: 0.466428982311007 and parameters: {'n_layers': 2, 'n_units_l0': 12, 'n_units_l1': 45, 'activation': 'tanh', 'learning_rate_init': 0.0006911876906485073, 'alpha': 0.0001353113656059966}. Best is trial 1 with value: 0.4279733053460041.
[I 2025-09-01 18:07:49,118] Trial 3 finished with value: 0.3909563830930383 and parameters: {'n_layers': 3, 

[I 2025-09-01 18:08:21,934] A new study created in memory with name: no-name-ba48c879-88db-4b47-a13d-7defe4295a31


[I 2025-09-01 18:08:21,841] Trial 49 finished with value: 0.39140340193105577 and parameters: {'n_layers': 2, 'n_units_l0': 21, 'n_units_l1': 21, 'activation': 'relu', 'learning_rate_init': 0.01988329572433608, 'alpha': 0.005554311881493205}. Best is trial 42 with value: 0.3867077053923536.


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-09-01 18:08:22,600] Trial 0 finished with value: 0.6361695291775866 and parameters: {'n_layers': 2, 'n_units_l0': 92, 'n_units_l1': 54, 'activation': 'relu', 'learning_rate_init': 0.004373348145529513, 'alpha': 5.509929987896079e-05}. Best is trial 0 with value: 0.6361695291775866.
[I 2025-09-01 18:08:23,132] Trial 1 finished with value: 0.6455899804496307 and parameters: {'n_layers': 3, 'n_units_l0': 49, 'n_units_l1': 98, 'n_units_l2': 25, 'activation': 'relu', 'learning_rate_init': 0.011625974485725542, 'alpha': 1.0309188270255394e-05}. Best is trial 0 with value: 0.6361695291775866.
[I 2025-09-01 18:08:23,809] Trial 2 finished with value: 0.747600286476942 and parameters: {'n_layers': 1, 'n_units_l0': 37, 'activation': 'relu', 'learning_rate_init': 0.00016078898037430063, 'alpha': 0.0001267351302425392}. Best is trial 0 with value: 0.6361695291775866.
[I 2025-09-01 18:08:24,540] Trial 3 finished with value: 0.6513159781028623 and parameters: {'n_layers': 3, 'n_units_l0': 97,